# API Data Ingestion for Data Engineering


## Why API Ingestion Matters

Most real-world data comes from APIs:

- Payments (Stripe)
- User data (Auth systems)
- Analytics (Google, Meta)

👉 Your job as a Data Engineer is to:
- Fetch data
- Validate it
- Store it
- Make it usable


## Basic API Call


In [ ]:
import requests

url = "https://jsonplaceholder.typicode.com/posts"

response = requests.get(url)

print(response.status_code)
data = response.json()

print(f"Fetched {len(data)} records")
print(data[:2])


## Handling Failures (IMPORTANT)


In [ ]:
def fetch_data(url):
    try:
        response = requests.get(url, timeout=5)

        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            return []

        return response.json()

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)
        return []


data = fetch_data(url)
print(len(data))


## Transforming API Data


In [ ]:
def transform_posts(posts):
    return [
        {
            "id": p["id"],
            "title": p["title"],
            "title_length": len(p["title"])
        }
        for p in posts
    ]


cleaned = transform_posts(data)
print(cleaned[:2])


## Save to JSON File


In [ ]:
import json

with open("posts.json", "w") as f:
    json.dump(cleaned, f, indent=4)

print("Data saved")


## Full Pipeline Function


In [ ]:
def run_pipeline():
    url = "https://jsonplaceholder.typicode.com/posts"

    raw = fetch_data(url)

    if not raw:
        print("No data fetched")
        return

    transformed = transform_posts(raw)

    with open("pipeline_output.json", "w") as f:
        json.dump(transformed, f, indent=4)

    print("Pipeline completed successfully")


run_pipeline()


## Add Logging (Production Mindset)


In [ ]:
import logging

logging.basicConfig(level=logging.INFO)


def run_pipeline_logged():
    logging.info("Starting pipeline")

    data = fetch_data(url)

    if not data:
        logging.error("No data fetched")
        return

    transformed = transform_posts(data)

    with open("output.json", "w") as f:
        json.dump(transformed, f)

    logging.info("Pipeline completed")


run_pipeline_logged()


## Practice

1. Fetch users API:
https://jsonplaceholder.typicode.com/users

2. Extract:
- name
- email

3. Save to file


## Assignment

Build a reusable ingestion system:

- Input: API URL
- Output: JSON file

**Features:**
- Retry on failure
- Logging
- Basic validation

**Bonus:**
- Add timestamp to output


## Interview Questions

1. How do you handle API failures?
2. What is retry logic?
3. How do you deal with rate limits?
4. How do you validate API data?


## What you just built

Your first end-to-end shape: **API → transform → store**.

You used: **ingestion**, **error handling**, **logging**, **pipeline design**.


---

**Next:** deeper **error handling + logging**, then a **production-style reusable pipeline**.

**Reality check:** if you’ve run this notebook top-to-bottom, you’re already practicing how engineers ship ingestion—not just toy scripts.


## Your tasks

- [ ] Run all cells once online; confirm `posts.json`, `pipeline_output.json`, and `output.json` are created (gitignored).
- [ ] **Practice:** Fetch `/users`, extract `name` + `email`, save to your own JSON file.
- [ ] **Assignment:** One reusable function `ingest(url, out_path)` with **logging**, **retry** (at least 3 attempts or backoff), and **basic validation** (e.g. non-empty list); **bonus:** add `ingested_at` UTC ISO timestamp wrapper object.
- [ ] Read interview answers aloud; add one real-world rate-limit tactic you’d use (sleep + Respect `Retry-After`, token bucket, etc.).
